# City Council Meetings & Agendas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/city-council-agendas.ipynb)

**What:** City council meetings and agenda items from 18 US cities via the official Legistar Web API. Meeting date and time, body, matter, status, action, vote result, agenda PDF and attachment links. Keyword filters, deduped alerts, 93 passing tests. $0.001 per item.

**Why:** run the [`city-council-agendas`](https://apify.com/transparent_meteorite/city-council-agendas) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/city-council-agendas) for every option.

In [2]:
run_input = {'clients': ['denver'], 'maxResults': 5, 'onlyNewSinceLastRun': False}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/city-council-agendas').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.city-council-agendas runId:3HaRP4mRncN3sqBiP] -> Status: RUNNING, Message: 


[apify.city-council-agendas runId:3HaRP4mRncN3sqBiP] -> 2026-10-13T22:17:04.734Z ACTOR: Pulling container image of build UGFEcRctNCEX0PX9I from registry.


[apify.city-council-agendas runId:3HaRP4mRncN3sqBiP] -> 2026-10-13T22:17:04.737Z ACTOR: Creating container.


[apify.city-council-agendas runId:3HaRP4mRncN3sqBiP] -> 2026-10-13T22:17:04.800Z ACTOR: Starting container.


[apify.city-council-agendas runId:3HaRP4mRncN3sqBiP] -> 2026-10-13T22:17:04.802Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.city-council-agendas runId:3HaRP4mRncN3sqBiP] -> 2026-10-13T22:17:07.921Z [agendas] Done. delivered=5 unsupported=0 errors=0


[apify.city-council-agendas runId:3HaRP4mRncN3sqBiP] -> 2026-10-13T22:17:07.923Z INFO  [Status message]: Done: 5 items


[apify.city-council-agendas runId:3HaRP4mRncN3sqBiP] -> 2026-10-13T22:17:07.944Z Thanks for using City Council Agendas API. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/ibCBXKSdYhek5QmMY/issues) with the run URL and we will reply there.


[apify.city-council-agendas runId:3HaRP4mRncN3sqBiP] -> Status: SUCCEEDED, Message: Done: 5 items


Run status: SUCCEEDED | items: 5


## 4. Tabulate

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["city", "body", "meetingDate", "agendaStatus", "title", "matterType", "meetingUrl"] if c in df.columns]
df[SHOW]

,city,body,meetingDate,agendaStatus,title,matterType,meetingUrl
0,"Denver, CO",Finance and Business,2026-10-06,Final,Consent Items,NaN,https://denver.legistar.com/MeetingDetail.aspx?LEGID=12976&GID=655...
1,"Denver, CO",Finance and Business,2026-10-06,Final,"Amends a Master Purchase Order with Greening Enterprises, Inc dba ...",Resolution,https://denver.legistar.com/MeetingDetail.aspx?LEGID=12976&GID=655...
2,"Denver, CO",Finance and Business,2026-10-06,Final,"A bill for an ordinance amending Ordinance No. 400, Series of 2008...",Bill,https://denver.legistar.com/MeetingDetail.aspx?LEGID=12976&GID=655...
3,"Denver, CO",Finance and Business,2026-10-06,Final,Approves Land Acquisition Ordinance granting the authority to acqu...,Bill,https://denver.legistar.com/MeetingDetail.aspx?LEGID=12976&GID=655...
4,"Denver, CO","Parks, Art and Culture",2026-10-06,Final,Attendance,NaN,https://denver.legistar.com/MeetingDetail.aspx?LEGID=12981&GID=655...


## 5. A quick analysis

In [5]:
print(df['body'].value_counts())
df['meetingDate'].value_counts().sort_index()

body
Finance and Business      4
Parks, Art and Culture    1
Name: count, dtype: int64


meetingDate
2026-10-06    5
Name: count, dtype: int64

## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [city-council-agendas on the Apify Store](https://apify.com/transparent_meteorite/city-council-agendas).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.